<a href="https://colab.research.google.com/github/BE7nHgA6/Cross-Market-Analysis-/blob/main/Cross_Market_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# pip install to install required Python libraries for data collection, analysis, and visualization
!pip install yfinance requests pandas matplotlib seaborn

In [2]:
# import required libraries to fetch data from APIs, process it using pandas, store it in SQLite database, and handle date operations.
import pandas as pd
import requests
import sqlite3
import yfinance as yf
from datetime import datetime

In [3]:
# use sqlite3.connect to connect to the database and cursor to execute SQL queries.
conn = sqlite3.connect("/content/market_analysis.db")
cursor = conn.cursor()

In [4]:

#This code creates 4 tables to store cryptocurrency, crypto prices, oil prices, and stock market data.
cursor.execute("""
CREATE TABLE IF NOT EXISTS cryptocurrencies (
    id TEXT PRIMARY KEY,
    symbol TEXT,
    name TEXT,
    current_price REAL,
    market_cap REAL,
    market_cap_rank INTEGER,
    total_volume REAL,
    circulating_supply REAL,
    total_supply REAL,
    ath REAL,
    atl REAL,
    date DATE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS crypto_prices (
    coin_id TEXT,
    date DATE,
    price_usd REAL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS oil_prices (
    date DATE PRIMARY KEY,
    price_usd REAL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS stock_prices (
    date DATE,
    open REAL,
    high REAL,
    low REAL,
    close REAL,
    volume REAL,
    ticker TEXT
)
""")

conn.commit()

In [5]:
#This code fetches top 250 cryptocurrencies data from CoinGecko API and converts it into a clean DataFrame

#Fetch crypto data from API
url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 250,
    "page": 1
}

response = requests.get(url, params=params)
data = response.json()

# Convert to DataFrame

crypto_df = pd.DataFrame(data)

#Select required columns

crypto_df = crypto_df[[
    "id","symbol","name","current_price","market_cap",
    "market_cap_rank","total_volume","circulating_supply",
    "total_supply","ath","atl","last_updated"
]]

# Convert date format

crypto_df["date"] = pd.to_datetime(crypto_df["last_updated"]).dt.date

# Drop unwanted column
crypto_df.drop(columns=["last_updated"], inplace=True)
# Handle missing values
crypto_df.fillna(0, inplace=True)

# Insert into database
crypto_df.to_sql("cryptocurrencies", conn, if_exists="replace", index=False)

# Preview data
crypto_df.head()



,id,symbol,name,current_price,market_cap,market_cap_rank,total_volume,circulating_supply,total_supply,ath,atl,date
0,bitcoin,btc,Bitcoin,75027.00,1501230862819,1,4.661644e+10,2.001793e+07,2.001793e+07,126080.00,67.810000,2026-04-19
1,ethereum,eth,Ethereum,2310.06,278703572931,2,1.382331e+10,1.206905e+08,1.206905e+08,4946.05,0.432979,2026-04-19
2,tether,usdt,Tether,1.00,186665040675,3,7.067615e+10,1.866121e+11,1.920791e+11,1.32,0.572521,2026-04-19
3,ripple,xrp,XRP,1.42,87493980450,4,2.357318e+09,6.156968e+10,9.998568e+10,3.65,0.002686,2026-04-19
4,binancecoin,bnb,BNB,619.90,83526192406,5,9.000319e+08,1.347867e+08,1.347867e+08,1369.99,0.039818,2026-04-19


In [6]:
crypto_df.to_sql("cryptocurrencies", conn, if_exists="replace", index=False)

250

In [7]:
top3 = crypto_df.nsmallest(3, "market_cap_rank")["id"].tolist()
top3

['bitcoin', 'ethereum', 'tether']

In [8]:
# fetched 1-year historical price data for top 3 cryptocurrencies using CoinGecko API, transformed timestamps into dates, and combined all data into a single DataFrame for analysis.
all_crypto_prices = pd.DataFrame()

for coin in top3:
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart"
    params = {"vs_currency": "usd", "days": 365}

    res = requests.get(url, params=params).json()

    prices = pd.DataFrame(res["prices"], columns=["timestamp","price"])
    prices["date"] = pd.to_datetime(prices["timestamp"], unit="ms").dt.date
    prices["coin_id"] = coin

    prices = prices[["coin_id","date","price"]]
    prices.columns = ["coin_id","date","price_usd"]

    all_crypto_prices = pd.concat([all_crypto_prices, prices])

all_crypto_prices.head()

,coin_id,date,price_usd
0,bitcoin,2025-04-20,85126.662443
1,bitcoin,2025-04-21,85073.165449
2,bitcoin,2025-04-22,87452.046991
3,bitcoin,2025-04-23,93576.165886
4,bitcoin,2025-04-24,93605.452309


In [9]:
# used pandas to_sql() function to store processed crypto price data into SQLite database, replacing existing data to maintain updated records.
all_crypto_prices.to_sql("crypto_prices", conn, if_exists="replace", index=False)

1098

In [10]:
#load oil price data from CSV, filter required dates, clean it, and prepare it for analysis
oil_url = "https://raw.githubusercontent.com/datasets/oil-prices/main/data/wti-daily.csv"

oil_df = pd.read_csv(oil_url)

oil_df["date"] = pd.to_datetime(oil_df["Date"]).dt.date
oil_df = oil_df[(oil_df["date"] >= pd.to_datetime("2020-01-01").date()) &
                (oil_df["date"] <= pd.to_datetime("2026-01-01").date())]

oil_df = oil_df[["date","Price"]]
oil_df.columns = ["date","price_usd"]

oil_df.head()

,date,price_usd
8569,2020-01-02,61.17
8570,2020-01-03,63.00
8571,2020-01-06,63.27
8572,2020-01-07,62.70
8573,2020-01-08,59.65


In [11]:
# used pandas to_sql() function to store cleaned oil price data into SQLite database for further analysis
oil_df.to_sql("oil_prices", conn, if_exists="replace", index=False)

1500

In [12]:
#used yfinance to download historical stock data for major indices, cleaned the dataset, and stored it in a structured format for analysis.
tickers = ["^GSPC", "^IXIC", "^NSEI"]

all_stocks = pd.DataFrame()

for ticker in tickers:
    df = yf.download(ticker, start="2020-01-01", end="2025-09-30")
    df.reset_index(inplace=True)

    df["ticker"] = ticker
    df["date"] = pd.to_datetime(df["Date"]).dt.date

    df = df[["date","Open","High","Low","Close","Volume","ticker"]]
    df.columns = ["date","open","high","low","close","volume","ticker"]

    all_stocks = pd.concat([all_stocks, df])

all_stocks.head()

/tmp/ipykernel_9158/1675575820.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2020-01-01", end="2025-09-30")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_9158/1675575820.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2020-01-01", end="2025-09-30")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_9158/1675575820.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2020-01-01", end="2025-09-30")
[*********************100%***********************]  1 of 1 completed


,date,open,high,low,close,volume,ticker
0,2020-01-02,3244.669922,3258.139893,3235.530029,3257.850098,3459930000,^GSPC
1,2020-01-03,3226.360107,3246.149902,3222.340088,3234.850098,3484700000,^GSPC
2,2020-01-06,3217.550049,3246.840088,3214.639893,3246.280029,3702460000,^GSPC
3,2020-01-07,3241.860107,3244.909912,3232.429932,3237.179932,3435910000,^GSPC
4,2020-01-08,3238.590088,3267.070068,3236.669922,3253.050049,3726840000,^GSPC


In [13]:
# stored cleaned stock market data into SQLite database using pandas to_sql() for further analysis and visualization
all_stocks.to_sql("stock_prices", conn, if_exists="replace", index=False)

4309

In [14]:
pd.read_sql("""
SELECT name, market_cap
FROM cryptocurrencies
ORDER BY market_cap DESC
LIMIT 3
""", conn)

,name,market_cap
0,Bitcoin,1501230862819
1,Ethereum,278703572931
2,Tether,186665040675


In [15]:
pd.read_sql("""
SELECT MAX(price_usd)
FROM crypto_prices
WHERE coin_id='bitcoin'
""", conn)

,MAX(price_usd)
0,124773.508231


In [16]:
pd.read_sql("""
SELECT MAX(price_usd) FROM oil_prices
""", conn)

,MAX(price_usd)
0,123.64


In [17]:
pd.read_sql("""
SELECT MAX(close)
FROM stock_prices
WHERE ticker='^IXIC'
""", conn)

,MAX(close)
0,22788.980469


In [18]:
pd.read_sql("""
SELECT c.date, c.price_usd AS btc_price,
       o.price_usd AS oil_price
FROM crypto_prices c
JOIN oil_prices o
ON c.date = o.date
WHERE c.coin_id = 'bitcoin'
LIMIT 10
""", conn)

,date,btc_price,oil_price
0,2025-04-21,85073.165449,63.48
1,2025-04-22,87452.046991,64.60
2,2025-04-23,93576.165886,62.64
3,2025-04-24,93605.452309,63.55
4,2025-04-25,93872.814229,63.85
5,2025-04-28,93809.337820,63.30
6,2025-04-29,95030.606455,61.84
7,2025-04-30,94256.359463,59.55
8,2025-05-01,94235.753310,60.59
9,2025-05-02,96426.945223,59.67


In [19]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 102.0 MB/s eta 0:00:00
--2026-04-19 09:59:22--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-19 09:59:22--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-19T10%3A53%3A09Z&rscd=attachment%3B+filename%3Dc

In [20]:
!pip install streamlit-option-menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 20.9 MB/s eta 0:00:00


In [22]:
!streamlit run /content/app.py &>/content/logs.txt &

In [23]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://morrison-none-governments-oak.trycloudflare.com


In [53]:
%%writefile app.py

import streamlit as st
import sqlite3
import pandas as pd

# ---------------------------
# PAGE CONFIG + STYLE
# ---------------------------
st.set_page_config(page_title="Cross Market Dashboard", layout="wide")



# ---------------------------
# DB CONNECTION
# ---------------------------
conn = sqlite3.connect("/content/market_analysis.db")

# ---------------------------
# SIDEBAR
# ---------------------------
st.sidebar.title("📌 Navigation")

page = st.sidebar.radio("Go to", [
    "📊 Market Overview",
    "🔍 SQL Query Runner",
    "💰 Top 3 Crypto Analysis"

])



# ---------------------------
# PAGE 1: MARKET OVERVIEW
# ---------------------------
if page == "📊 Market Overview":
    st.markdown("""
    <style>

    .stApp {
        background-color: #F38BC8;
    }
    h1, h2, h3 {
        font-family: 'Arial';
        color: #2c3e50;
    }
    </style>

""", unsafe_allow_html=True)

    st.title("📊 Cross-Market Overview")
    st.badge("Crypto oil stock market|SQL powered analytics")

    # Date range
    min_date = pd.read_sql("SELECT MIN(date) FROM crypto_prices", conn).iloc[0,0]
    max_date = pd.read_sql("SELECT MAX(date) FROM crypto_prices", conn).iloc[0,0]

    col1, col2 = st.columns(2)
    start_date = col1.date_input("Start Date", pd.to_datetime(min_date))
    end_date = col2.date_input("End Date", pd.to_datetime(max_date))

    # Averages
    def safe(df):
        val = df.iloc[0,0]
        return round(val,2) if val else 0

    btc = pd.read_sql(f"""
        SELECT AVG(price_usd) FROM crypto_prices
        WHERE coin_id='bitcoin'
        AND date BETWEEN '{start_date}' AND '{end_date}'
    """, conn)

    oil = pd.read_sql(f"""
        SELECT AVG(price_usd) FROM oil_prices
        WHERE date BETWEEN '{start_date}' AND '{end_date}'
    """, conn)

    sp = pd.read_sql(f"""
        SELECT AVG(close) FROM stock_prices
        WHERE ticker='^GSPC'
        AND date BETWEEN '{start_date}' AND '{end_date}'
    """, conn)

    nifty = pd.read_sql(f"""
        SELECT AVG(close) FROM stock_prices
        WHERE ticker='^NSEI'
        AND date BETWEEN '{start_date}' AND '{end_date}'
    """, conn)

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("₿ Bitcoin Avg", safe(btc))
    c2.metric("🟦 Oil Avg", safe(oil))
    c3.metric("📈 S&P 500 Avg", safe(sp))
    c4.metric("IN NIFTY Avg", safe(nifty))

    # SNAPSHOT TABLE (FIXED)
    st.subheader("📋 Daily Market Snapshot")

    snapshot = pd.read_sql(f"""
        SELECT c.date,
               c.price_usd AS bitcoin_price,
               o.price_usd AS oil_price,
               s.close AS sp500,
               n.close AS nifty
        FROM crypto_prices c
        LEFT JOIN oil_prices o ON c.date = o.date
        LEFT JOIN stock_prices s ON c.date = s.date AND s.ticker='^GSPC'
        LEFT JOIN stock_prices n ON c.date = n.date AND n.ticker='^NSEI'
        WHERE c.coin_id='bitcoin'
        AND c.date BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY c.date DESC
    """, conn)

    # REMOVE NONE VALUES
    snapshot = snapshot.dropna()

    st.dataframe(snapshot, use_container_width=True)

# ---------------------------
# PAGE 2: SQL QUERY RUNNER
# ---------------------------
elif page == "🔍 SQL Query Runner":

    st.title("🔍 SQL Query Runner")
    st.badge("Predefined analytical SQL queries")
    st.markdown("""
        <style>
        .stApp {
            background-color: #63e5ff;
        }
        </style>
    """, unsafe_allow_html=True)

    query_option = st.selectbox("Select Query", [

# ------------------ CRYPTOCURRENCIES ------------------
"Top 3 Cryptos by Market Cap",
"Circulating Supply > 90%",
"Coins Near ATH",
"Avg Rank (Volume > 1B)",
"Most Recently Updated Coin",

# ------------------ CRYPTO PRICES ------------------
"Bitcoin Highest Price (1 Year)",
"Ethereum Avg Price (1 Year)",
"Bitcoin Trend Jan",
"Highest Avg Price Coin",
"Bitcoin % Change",

# ------------------ OIL ------------------
"Highest Oil Price (5 Years)",
"Oil Avg Per Year",
"Oil During COVID",
"Lowest Oil Price",
"Oil Volatility Per Year",

# ------------------ STOCKS ------------------
"All Prices for Ticker",
"NASDAQ Highest Close",
"S&P Top 5 Volatile Days",
"Monthly Avg Closing",
"NSEI Avg Volume 2024",

# ------------------ JOINS ------------------
"Bitcoin vs Oil Avg",
"Bitcoin vs S&P",
"Ethereum vs NASDAQ",
"Oil Spike vs Bitcoin",
"Top 3 Crypto vs Nifty",
"S&P vs Oil",
"BTC vs Oil Correlation",
"NASDAQ vs Ethereum",
"Top 3 Crypto Join Stocks",
"Multi Market Comparison"
])
    queries = {

# ------------------ CRYPTOCURRENCIES ------------------

"Top 3 Cryptos by Market Cap":
"""
SELECT name, market_cap
FROM cryptocurrencies
ORDER BY market_cap DESC LIMIT 3
""",

"Circulating Supply > 90%":
"""
SELECT name FROM cryptocurrencies
WHERE circulating_supply >= 0.9 * total_supply
""",

"Coins Near ATH":
"""
SELECT name FROM cryptocurrencies
WHERE current_price >= 0.9 * ath
""",

"Avg Rank (Volume > 1B)":
"""
SELECT AVG(market_cap_rank)
FROM cryptocurrencies
WHERE total_volume > 1000000000
""",


"Most Recently Updated Coin":
"""
SELECT name FROM cryptocurrencies
ORDER BY market_cap DESC LIMIT 1
""",


# ------------------ CRYPTO PRICES ------------------

"Bitcoin Highest Price (1 Year)":
"""
SELECT MAX(price_usd)
FROM crypto_prices
WHERE coin_id='bitcoin'
""",

"Ethereum Avg Price (1 Year)":
"""
SELECT AVG(price_usd)
FROM crypto_prices
WHERE coin_id='ethereum'
""",

"Bitcoin Trend Jan":
"""
SELECT date, price_usd
FROM crypto_prices
WHERE coin_id='bitcoin' AND strftime('%m', date)='01'
""",

"Highest Avg Price Coin":
"""
SELECT coin_id, AVG(price_usd) avg_price
FROM crypto_prices
GROUP BY coin_id
ORDER BY avg_price DESC LIMIT 1
""",

"Bitcoin % Change":
"""
SELECT
((MAX(price_usd)-MIN(price_usd))*100.0/MIN(price_usd)) AS pct_change
FROM crypto_prices
WHERE coin_id='bitcoin'
""",

# ------------------ OIL ------------------

"Highest Oil Price (5 Years)":
"""
SELECT MAX(price_usd) FROM oil_prices
""",

"Oil Avg Per Year":
"""
SELECT strftime('%Y', date) year, AVG(price_usd)
FROM oil_prices GROUP BY year
""",

"Oil During COVID":
"""
SELECT * FROM oil_prices
WHERE date BETWEEN '2020-03-01' AND '2020-04-30'
""",

"Lowest Oil Price":
"""
SELECT MIN(price_usd) FROM oil_prices
""",

"Oil Volatility Per Year":
"""
SELECT strftime('%Y', date) year,
MAX(price_usd)-MIN(price_usd) volatility
FROM oil_prices GROUP BY year
""",

# ------------------ STOCKS ------------------

"All Prices for Ticker":
"""
SELECT * FROM stock_prices
WHERE ticker='^GSPC'
""",

"NASDAQ Highest Close":
"""
SELECT MAX(close)
FROM stock_prices
WHERE ticker='^IXIC'
""",

"S&P Top 5 Volatile Days":
"""
SELECT date, high-low diff
FROM stock_prices
WHERE ticker='^GSPC'
ORDER BY diff DESC LIMIT 5
""",

"Monthly Avg Closing":
"""
SELECT ticker, strftime('%Y-%m', date) month,
AVG(close)
FROM stock_prices GROUP BY ticker, month
""",

"NSEI Avg Volume 2024":
"""
SELECT AVG(volume)
FROM stock_prices
WHERE ticker='^NSEI' AND strftime('%Y',date)='2024'
""",

# ------------------ JOINS ------------------

"Bitcoin vs Oil Avg":
"""
SELECT AVG(c.price_usd), AVG(o.price_usd)
FROM crypto_prices c
JOIN oil_prices o ON c.date=o.date
WHERE c.coin_id='bitcoin'
""",

"Bitcoin vs S&P":
"""
SELECT c.date, c.price_usd, s.close
FROM crypto_prices c
JOIN stock_prices s ON c.date=s.date
WHERE c.coin_id='bitcoin' AND s.ticker='^GSPC'
""",

"Ethereum vs NASDAQ":
"""
SELECT c.date, c.price_usd, s.close
FROM crypto_prices c
JOIN stock_prices s ON c.date=s.date
WHERE c.coin_id='ethereum' AND s.ticker='^IXIC'
""",

"Oil Spike vs Bitcoin":
 """
 SELECT
    o.date,
    o.price_usd AS oil_price,
    c.price_usd AS bitcoin_price
FROM oil_prices o
JOIN crypto_prices c ON o.date = c.date
WHERE c.coin_id = 'bitcoin'
ORDER BY o.price_usd DESC
LIMIT 10
""",


"Top 3 Crypto vs Nifty":
"""
SELECT c.date, c.coin_id, c.price_usd, s.close
FROM crypto_prices c
JOIN stock_prices s ON c.date=s.date
WHERE s.ticker='^NSEI'
""",

"S&P vs Oil":
"""
SELECT s.date, s.close, o.price_usd
FROM stock_prices s
JOIN oil_prices o ON s.date=o.date
WHERE s.ticker='^GSPC'
""",

"BTC vs Oil Correlation":
"""
SELECT
    c.price_usd AS bitcoin_price,
    o.price_usd AS oil_price
FROM crypto_prices c
JOIN oil_prices o ON c.date = o.date
WHERE c.coin_id = 'bitcoin'
""",

"NASDAQ vs Ethereum":
"""
SELECT s.date, s.close, c.price_usd
FROM stock_prices s
JOIN crypto_prices c ON s.date=c.date
WHERE s.ticker='^IXIC' AND c.coin_id='ethereum'
""",

"Top 3 Crypto Join Stocks":
"""
SELECT c.date, c.coin_id, c.price_usd, s.close
FROM crypto_prices c
JOIN stock_prices s ON c.date=s.date
""",

"Multi Market Comparison":
"""
SELECT
    c.date,
    c.price_usd AS bitcoin_price,
    o.price_usd AS oil_price,
    s.close AS sp500
FROM crypto_prices c
JOIN oil_prices o ON c.date = o.date
JOIN stock_prices s ON c.date = s.date
WHERE c.coin_id = 'bitcoin'
"""
}
    if st.button("Run Query"):
        result = pd.read_sql(queries[query_option], conn)
        st.success("✅ Query executed successfully")
        st.dataframe(result)

# ---------------------------
# PAGE 3: TOP 3 CRYPTO
# ---------------------------
elif page == "💰 Top 3 Crypto Analysis":
    st.markdown("""
        <style>
        .stApp {
            background-color:#C0AFE2;
        }
        </style>
    """, unsafe_allow_html=True)
    st.title("💰 Top 3 Crypto Analysis")
    st.badge("Daily price analysis for top cryptocurrencies")

    coins = pd.read_sql("""
        SELECT id FROM cryptocurrencies
        ORDER BY market_cap_rank ASC
        LIMIT 3
    """, conn)

    coin_list = coins["id"].tolist()

    if not coin_list:
        st.error("No coins found")
        st.stop()

    selected_coin = st.selectbox("Select Cryptocurrency", coin_list)

    min_date = pd.read_sql("SELECT MIN(date) FROM crypto_prices", conn).iloc[0,0]
    max_date = pd.read_sql("SELECT MAX(date) FROM crypto_prices", conn).iloc[0,0]

    col1, col2 = st.columns(2)
    start_date = col1.date_input("Start Date", pd.to_datetime(min_date))
    end_date = col2.date_input("End Date", pd.to_datetime(max_date))

    df = pd.read_sql(f"""
        SELECT date, price_usd
        FROM crypto_prices
        WHERE coin_id='{selected_coin}'
        AND date BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY date
    """, conn)

    if df.empty:
        st.warning("No data available")
    else:
        df["date"] = pd.to_datetime(df["date"])

        st.subheader(f"📈 {selected_coin.upper()} Price Trend")
        st.bar_chart(df.set_index("date"))

        st.subheader("📊 Data Table")
        st.dataframe(df, use_container_width=True)

Overwriting app.py
